# Drive, github and requirements settings

In [ ]:
from google.colab import drive
from google.colab import userdata


drive.mount('/content/drive')
KEY = userdata.get('KEY')

Mounted at /content/drive


Mount colab to google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


set Github access token

In [ ]:
import os
from google.colab import userdata
github_access_token = userdata.get('GITHUB_TOKEN')

# Replace 'your_token_here' with your actual token and 'your_repo_url_here' with your repository URL
os.environ['GITHUB_TOKEN'] = github_access_token

repo_url = 'https://github.com/sustaz/principle_of_law_detection.git'
modified_url = repo_url.replace('https://', f'https://{os.environ["GITHUB_TOKEN"]}@')

Clone Repository

In [ ]:
!git clone {modified_url}

Cloning into 'principle_of_law_detection'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 38 (delta 16), reused 26 (delta 7), pack-reused 0
Receiving objects: 100% (38/38), 120.39 KiB | 2.94 MiB/s, done.
Resolving deltas: 100% (16/16), done.


Set github credentials

In [ ]:
!chmod +x ./principle_of_law_detection/bash_commands/set_git_credentials.sh
!./principle_of_law_detection/bash_commands/set_git_credentials.sh

Syncronize notebook version

In [ ]:
!cp /content/drive/MyDrive/POLINE/gpt_notebook.ipynb /content/principle_of_law_detection/

In [ ]:
!cp /content/drive/MyDrive/POLINE/old_test_outputs.ipynb /content/principle_of_law_detection/

Add, commit, push code

In [ ]:
!chmod +x ./principle_of_law_detection/bash_commands/add_commit_push.sh
!./principle_of_law_detection/bash_commands/add_commit_push.sh "cleaned gpt notebook and built library"

[main df8882e] cleaned gpt notebook and built library
 7 files changed, 2 insertions(+), 19 deletions(-)
 rewrite gpt_notebook.ipynb (97%)
 rewrite old_test_outputs.ipynb (88%)
 rename {utils => src}/evaluation.py (100%)
 rename {utils => src}/gpt_utils.py (100%)
 rename {utils => src}/text_preprocessing.py (100%)
 rename utils/save_results.py => src/utils.py (100%)
 delete mode 100644 utils/utils.py
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 5.95 KiB | 2.97 MiB/s, done.
Total 9 (delta 3), reused 1 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/sustaz/principle_of_law_detection.git
   2616a1d..df8882e  main -> main


Install requirements

In [ ]:
!pip install -r '/content/principle_of_law_detection/requirements.txt'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.4/327.4 kB 23.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.9 MB/s eta 0:00:00


# Import libraries

In [ ]:
from principle_of_law_detection.src import gpt_utils as gu, utils as sr, text_preprocessing as tp, evaluation as ev
import json
import os
import pandas as pd

# Single experiments

In [ ]:
file = "/content/drive/MyDrive/POLINE/Dataset_V1/Judgements_Subset/ELVOSPOL s.r.o. v Odvolací finanční ředitelství.txt.xml"

with open(file) as f:
    file = f.read()

txt = "".join(file)

txt = tp.extract_text_between_markers(txt)

response = gu.ask_gpt_2(prompt_single_trial(txt), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

# MASSIVE EXPERIMENTS

In [ ]:
def jpol_prompt(txt):
  prompt =  f"""" Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.

    {txt}

    A JPOL (Judicial Principle of Law) should:

    Source and Content:
        Be a portion of text extracted from the argumentative part of a judgment.
        Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

    Nature of Interpretation:
        Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
        Not be a rephrase of the legislation.

    Citations and Endorsements:
        A JPOL can cite another JPOL.
        A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

    Use the following format for the output:
      Paragraph number: Y if JPOL.
      Paragraph number: N if not JPOL."""

  return prompt


system_prompt = "You are a professionist judge with strong knowledge on jurisdiction and tax law, expert about Judicial Principles of Law (JPOLs) from legal judgments. "

In [ ]:
jsons_root = "/content/drive/MyDrive/POLINE/Annotazioni/poline_jsons/"
annotations_files = os.listdir(jsons_root)

In [ ]:
judgements_root = "/content/drive/MyDrive/POLINE/Dataset/Dataset_V1/Judgements_Subset"
judgemnts = os.listdir(judgements_root)

In [ ]:
prompt_name = "Piera_e_Alessia 24.06.24_1"
responses = []

for idx, judgement in enumerate(judgemnts):


  with open(os.path.join(judgements_root, judgement)) as f:
    file = f.read()

  txt = "".join(file)

  txt = tp.extract_text_between_markers(txt)

  response = gu.ask_gpt_2(jpol_prompt(txt), system_prompt, KEY, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

  sr.write_text_to_docx(response, f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/{prompt_name}_{judgement}_full_response.docx")

  responses.append((response, judgement))

Save results

In [ ]:
dfs = []
for response, file_name in responses:
    dfs.append((sr.extract_paragraphs(response), file_name[:5]))

results_df = sr.concatenate_dataframes(dfs)
results_df.to_excel(f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/{prompt_name}_remaining.xlsx", index=False)

Read GT

In [ ]:
ground_truths = []

for ann_file in annotations_files:

  ann_dict = json.load(open(os.path.join(jsons_root,ann_file)))


  for ann in ann_dict['annotations']:
    paragraph_number = ann['text'].split()[0]
    # split_par = str(int(paragraph_number) + 1)
    # txt_par = txt.split(split_par)
    label = ann['type']
    file_name = ann_file[:5]

    ground_truths.append((file_name, paragraph_number, label))

ground_truth_df = pd.DataFrame(ground_truths, columns=['file_name', 'paragraph_number', 'ground_truth'])

# EVALUATION

In [ ]:
#results_df = pd.read_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/p_aspries_2_predictions.xlsx")

# Chiamare la funzione
metrics_by_filename, merged_df = ev.compute_metrics(ground_truth_df, results_df)

# Mostrare i risultati
print(metrics_by_filename)

precision, recall, f1 = ev.compute_total_metrics(ground_truth_df, results_df)

print(f'<--------------------->')

print(f'Total Precision: {precision:.2f}')
print(f'Total Recall: {recall:.2f}')
print(f'Total F1-Score: {f1:.2f}')

   file_name  precision    recall        f1
0      A & G   0.916667  1.000000  0.956522
1      Autor   0.727273  0.888889  0.800000
2      Boehr   0.636364  1.000000  0.777778
3      CS an   0.800000  0.923077  0.857143
4      DNB B   0.933333  0.875000  0.903226
5      ELVOS   0.500000  1.000000  0.666667
6      Euler   0.636364  1.000000  0.777778
7      Finan   0.583333  0.875000  0.700000
8      I Gmb   0.465116  0.952381  0.625000
9      Micha   0.571429  1.000000  0.727273
10     Minis   0.777778  1.000000  0.875000
<--------------------->
Total Precision: 0.64
Total Recall: 0.95
Total F1-Score: 0.77


In [ ]:
metrics_by_filename.to_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/metrics_evaluation/p_aspries_2_predictions.xlsx")

- controllare qual è il valore di f1 dati solo gli esempi veri



- Provare predizioni di 5 in 5 paragrafi

- Creare un file univoco di predizioni, ground truth e testo per error analysis

In [ ]:
def extract_text_between_paragraphs(text, prev_par, next_par):
    # Define the patterns to search for
    pattern1 = re.compile(f"{prev_par}(.*?){next_par}", re.DOTALL)

    # Try to find matches for both patterns
    match1 = pattern1.search(text)

    # Return the matched text if found
    if match1:
        return match1.group(1).strip()
    else:
        return None

In [ ]:
par_txt = []
import re

for idx, row in merged_df.iterrows():

  text_file = [filename for filename in judgemnts if filename.startswith(row['file_name'])][0]

  with open(os.path.join(judgements_root, text_file)) as f:
    file = f.read()

  txt = "".join(file)

  txt = tp.extract_text_between_markers(txt).replace("\n", " ")
  txt = extract_text_between_paragraphs(txt, str(int(row['paragraph_number'])), str(int(row['paragraph_number']) + 1) )

  try:
    txt_par = txt.split(split_par)[0]
  except:
    print(str(int(row['paragraph_number'])))
    continue

  par_txt.append((row['file_name'], row['paragraph_number'], txt_par))

31
42
66
56
45
45
47
44
83
71


In [ ]:
par_txt_df = pd.DataFrame(par_txt, columns=['file_name', 'paragraph_number', 'text'])

In [ ]:
pd.merge(par_txt_df, merged_df, on=['file_name', 'paragraph_number']).to_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/p_aspries_2_pred+gt.xlsx")